# Reading the dataset

In [1]:
import pandas as pd
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from scipy.sparse import csr_matrix

In [2]:
features_df = pd.read_csv("../data/features_df.csv")
edgelist = pd.read_csv("../data/edgelist.csv")

In [43]:
edgelist_full = pd.read_csv("../data/edgelist_full.csv")

In [46]:
edgelist_full.head()

,user_id,0000119c-0f93-4e28-894b-14d32aa2ed2e,00004b7e-33e5-43d2-a1e3-a089459a23a9,0000a92b-f14b-4f08-b175-3d456ffe0d65,00010059-d70f-490b-b4dc-e9dcf4a51377,000136a6-a047-4ac3-beaa-c0e618879a71,000143c0-f6f0-4a8c-a799-6a2e807e6840,00014733-1924-47ef-8095-951704934e54,00014b37-d53c-46b5-a59c-35261927fbc0,00015d72-6442-4f87-bed0-3be25360daac,...,1d15a10e-c0b0-4ee2-9b1c-77c2f136890f,1d15b4f7-cb6a-426b-acc9-3cd79a277fbf,1d15c599-0978-439b-ba94-6284c9d93776,1d164300-13d9-43dd-959f-e3b587b187b0,1d165855-f4ec-467a-b13c-fbc5030b1b0f,1d166cc7-e670-4c22-a4a2-d3e5198eb8a2,1d166fb2-f5a7-44f0-b534-f0eb673749b4,1d168ec1-29f0-4677-9985-44079d1281a5,1d1697ce-b310-4d07-9b7d-26ea332e5d98,1d16cfaa-7cd2-4fff-87b1-1bcb5c225325
0,0000119c-0f93-4e28-894b-14d32aa2ed2e,1.000000,0.911684,0.638658,0.815151,0.655604,0.724772,0.922397,0.970891,0.872202,...,0.901866,0.457812,0.223114,0.789606,0.831909,0.901401,0.741188,0.957292,0.984855,0.607844
1,00004b7e-33e5-43d2-a1e3-a089459a23a9,0.911684,1.000000,0.608216,0.777455,0.851311,0.717544,0.861165,0.888245,0.832681,...,0.812725,0.459330,0.314416,0.878546,0.820786,0.929189,0.813069,0.886230,0.895812,0.629115
2,0000a92b-f14b-4f08-b175-3d456ffe0d65,0.638658,0.608216,1.000000,0.403284,0.212567,0.638991,0.847080,0.467534,0.795937,...,0.329368,0.698454,0.246598,0.741526,0.593330,0.596892,0.795110,0.430791,0.573829,0.883425
3,00010059-d70f-490b-b4dc-e9dcf4a51377,0.815151,0.777455,0.403284,1.000000,0.592973,0.761170,0.643833,0.822318,0.689520,...,0.838698,0.483862,0.538481,0.594784,0.641031,0.800343,0.653766,0.756768,0.805076,0.532091
4,000136a6-a047-4ac3-beaa-c0e618879a71,0.655604,0.851311,0.212567,0.592973,1.000000,0.383687,0.504146,0.709116,0.452896,...,0.708031,0.167438,0.078363,0.609366,0.514946,0.694992,0.545531,0.747200,0.675614,0.211096


In [52]:
edgelist_full.set_index("user_id", inplace=True)
flat_df = edgelist_full.stack().reset_index()
flat_df.columns = ["user_id_anchor", "user_id_match", "cosine_score"]

In [54]:
flat_df.shape

(95902849, 3)

In [3]:
edgelist.head()

,user_1,user_2,similarity
0,148740aa-6c1a-48e5-a611-668ca3059d7f,1650061f-f8b4-4ac1-b160-db91b0a23d54,1.0
1,10018723-a86e-470a-9d93-817206368f00,145f8214-e3a7-4139-8b48-9d65fac9ad29,1.0
2,122d22c2-1354-4682-ae81-52c9e936f4ff,15a5981a-eb8b-43d2-9499-6cef65d8545a,1.0
3,13359dcf-94e2-4f14-9f91-df645a1150fb,15dce074-3c6c-41e7-852c-a75472f35f5c,1.0
4,121a4039-8ba5-45a2-bdee-7ffc8f0eddce,13ed7896-4fde-49f3-af62-9e7875cad325,1.0


# Pre-processing

In [4]:
# save user ids for mapping
user_ids = features_df["user_id"].values

# one-hot encode all categorical
cat_cols = features_df.select_dtypes(include=["object", "bool"]).columns.tolist()
if "user_id" in cat_cols:
    cat_cols.remove("user_id")
df_numeric = features_df.drop(columns=["user_id"])
df_numeric = pd.get_dummies(df_numeric, columns=cat_cols)

# fillna with median
df_numeric = df_numeric.fillna(df_numeric.median())

# Standardize
scaler = StandardScaler()
X = scaler.fit_transform(df_numeric)

/var/folders/jh/9_qy7nd96v9_q0_x1sffyq9c0000gn/T/ipykernel_2382/1985231353.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = features_df.select_dtypes(include=["object", "bool"]).columns.tolist()


# Build Autoencoder

## set model

### first try

In [5]:
input_dim = X.shape[1]
encoding_dim = 32  # hyperparam

input_layer = Input(shape=(input_dim,))

# Encoder
encoded = Dense(64, activation="relu")(input_layer)
encoded = Dense(encoding_dim, activation="relu")(encoded)  # user embedding

# Decoder
decoded = Dense(64, activation="relu")(encoded)
decoded = Dense(input_dim, activation="linear")(decoded)  # restore original dim

In [6]:
autoencoder = Model(inputs=input_layer, outputs=decoded)
encoder = Model(inputs=input_layer, outputs=encoded)  # extract Encoder for embedding

autoencoder.compile(optimizer="adam", loss="mse")

In [ ]:
# Train + Eval
# hyperparam
history = autoencoder.fit(
    X, X, epochs=50, batch_size=32, validation_split=0.2, verbose=1
)

# evaluate MSE between decoded & input
mse_loss = autoencoder.evaluate(X, X, verbose=0)
print(f"\nOverall Reconstruction MSE Loss: {mse_loss:.4f}")

Means we lost 15.56% of original info when converted 110+ features to 32 (hopefully
noises we wish to lose).

In [8]:
user_embeddings = encoder.predict(X)

# evaluate cosine similarity
similarity_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(similarity_matrix, index=user_ids, columns=user_ids)

61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 729us/step


## Evals

In [34]:
import importlib
import evaluation as evals

importlib.reload(evals)


<module 'evaluation' from '/Users/sanch/workspace/spotify-app/training/evaluation.py'>

In [35]:
k = 50
match_generated_for = similarity_df.index.nunique()
universe_size = edgelist.user_1.nunique()
print(f"Similarity DF user count: {match_generated_for}")
print(f"Edgelist DF user count: {universe_size}")
per_user, summary = evals.evaluate_topk(
    similarity_df,
    edgelist,
    k=k,
    anchor_col="user_1",
    other_col="user_2",
    score_col="similarity",
)
print("Total count of users we found a 'good' match:", per_user.shape[0])
good_matches = per_user.loc[per_user.intersection > 0, :].shape[0]
print(
    f"Pct users we found a 'good' match: {(good_matches / match_generated_for) * 100:.2f}%"
)
# for k, v in summary.items():
#     print(f"{k}: {v:.2f}")

Similarity DF user count: 1939
Edgelist DF user count: 9772
Total count of users we found a 'good' match: 975
Pct users we found a 'good' match: 42.03%


In [19]:
# --- User-level Recall@10 Evaluation ---

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

unique_anchors = valid_edgelist["user_id_anchor"].unique()

user_hits = 0
total_users = len(unique_anchors)

print(f"Starting User-level Evaluation for {total_users} users...")

for anchor in unique_anchors:
    positives = set(
        valid_edgelist[valid_edgelist["user_id_anchor"] == anchor]["user_id_positive"]
    )

    top_10_similar = set(similarity_df.loc[anchor].drop(anchor).nlargest(10).index)

    # any hit within top 10 counts as a hit for this user
    if len(positives.intersection(top_10_similar)) > 0:
        user_hits += 1

user_recall_rate = user_hits / total_users

print("\n[User-level Results]")
print(f"User-level Recall@10: {user_recall_rate:.2%}")
print(f"Successfully matched {user_hits} out of {total_users} users.")

Starting User-level Evaluation for 1931 users...

[User-level Results]
User-level Recall@10: 10.67%
Successfully matched 206 out of 1931 users.


### second try

In [40]:
input_dim = X.shape[1]
encoding_dim = 64  # larger than first model to retain more info

input_layer = Input(shape=(input_dim,))
# Encoder
x = Dense(128, activation="relu")(input_layer)
x = BatchNormalization()(x)  # added batch normalization
x = Dropout(0.1)(x)  # added dropout
encoded = Dense(encoding_dim, activation="relu")(x)

# Decoder
x = Dense(128, activation="relu")(encoded)
x = BatchNormalization()(x)
decoded = Dense(input_dim, activation="linear")(x)

autoencoder = Model(input_layer, decoded)
encoder = Model(input_layer, encoded)

autoencoder.compile(optimizer="adam", loss="mae")  # changed to MAE might be more r

# increase epochs
autoencoder.fit(X, X, epochs=100, batch_size=64, validation_split=0.1, verbose=0)

# Evaluate
print("Extracting embeddings and calculating similarity...")
user_embeddings = encoder.predict(X)
sim_matrix = cosine_similarity(user_embeddings)

sim_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

# new evaluation
k = 10
match_generated_for = sim_df.index.nunique()
universe_size = edgelist.user_1.nunique()
print(f"Similarity DF user count: {match_generated_for}")
print(f"Edgelist DF user count: {universe_size}")
per_user, summary = evals.evaluate_topk(
    similarity_df,
    edgelist,
    k=k,
    anchor_col="user_1",
    other_col="user_2",
    score_col="similarity",
)
good_matches = per_user.loc[per_user.intersection > 0, :].shape[0]
print("Total count of users we found a 'good' match:", good_matches)
print(
    f"Pct users we found a 'good' match: {(good_matches / match_generated_for) * 100:.2f}%"
)
# for k, v in summary.items():
#     print(f"{k}: {v:.2f}")


# ensure we're getting users in both df
# valid_edgelist = edgelist[
#     edgelist["user_id_anchor"].isin(user_ids)
#     & edgelist["user_id_positive"].isin(user_ids)
# ]

# print(f"Testing on {len(valid_edgelist)} pairs...")

# # get top 10 match from encoder
# hits = 0
# for anchor in valid_edgelist["user_id_anchor"].unique():
#     # get positives of this anchor in edgelist
#     positives = set(
#         valid_edgelist[valid_edgelist["user_id_anchor"] == anchor]["user_id_positive"]
#     )

#     # get this anchor's top 11 similarity (cuz self might be top 1)
#     top_n = sim_df.loc[anchor].nlargest(11).index.tolist()
#     if anchor in top_n:
#         top_n.remove(anchor)
#     top_10 = set(top_n[:10])

#     # any positive hit would count as one hit
#     if len(positives.intersection(top_10)) > 0:
#         hits += 1

# final_score = hits / len(valid_edgelist["user_id_anchor"].unique())
# print(f"New User-level Recall@10: {final_score:.2%}")

Extracting embeddings and calculating similarity...
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 947us/step
Similarity DF user count: 1939
Edgelist DF user count: 9772
Total count of users we found a 'good' match: 131
Pct users we found a 'good' match: 6.76%


In [42]:
per_user.groupby("intersection")["user_id"].nunique()

intersection
0    844
1    111
2     18
3      2
Name: user_id, dtype: int64

In [13]:
# Pair level recall：
user_embeddings = encoder.predict(X)
sim_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

print(f"Starting Pair-level Evaluation on {len(valid_edgelist)} pairs...")

hits = 0
total_pairs = len(valid_edgelist)

for idx, row in valid_edgelist.iterrows():
    anchor = row["user_id_anchor"]
    positive = row["user_id_positive"]

    top_10_similar = similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()

    if positive in top_10_similar:
        hits += 1

pair_level_recall = hits / total_pairs

print("\n[Final Results - Pair-level]")
print(
    f"Deep Autoencoder Reconstruction Loss (MAE): {autoencoder.evaluate(X, X, verbose=0):.4f}"
)
print(f"Pair-level Recall@10: {pair_level_recall:.2%}")
print(f"Successfully found {hits} out of {total_pairs} preset pairs.")

61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Starting Pair-level Evaluation on 13335 pairs...

[Final Results - Pair-level]
Deep Autoencoder Reconstruction Loss (MAE): 0.0298
Pair-level Recall@10: 1.90%
Successfully found 253 out of 13335 preset pairs.


### third try

In [9]:
from sklearn.preprocessing import MinMaxScaler  # 换成了 MinMaxScaler

# changed to use min max scaler
scaler = MinMaxScaler()
X = scaler.fit_transform(df_numeric)

# 2. deeper Autoencoder
input_dim = X.shape[1]
encoding_dim = 64  # Bottleneck dim

input_layer = Input(shape=(input_dim,))

# Input -> 256 -> 128 -> 64
# Encoder
encoder_layers = Sequential(
    [
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dense(encoding_dim, activation="relu"),  # Latent Space
    ]
)

# Decoder
decoder_layers = Sequential(
    [
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dense(
            input_dim, activation="sigmoid"
        ),  # cuz input is now 0-1，so use sigmoid for output
    ]
)

encoded_repr = encoder_layers(input_layer)
decoded_repr = decoder_layers(encoded_repr)

autoencoder = Model(inputs=input_layer, outputs=decoded_repr)
encoder_model = Model(inputs=input_layer, outputs=encoded_repr)

autoencoder.compile(optimizer="adam", loss="mae")  # MAE

# increased epochs
autoencoder.fit(X, X, epochs=100, batch_size=64, validation_split=0.1, verbose=1)

# eval
user_embeddings = encoder_model.predict(X)
sim_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

hits = 0
unique_anchors = valid_edgelist["user_id_anchor"].unique()

for anchor in unique_anchors:
    positives = set(
        valid_edgelist[valid_edgelist["user_id_anchor"] == anchor]["user_id_positive"]
    )

    top_10 = similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()

    if any(p in top_10 for p in positives):
        hits += 1

recall_at_10 = hits / len(unique_anchors)
print("\n[Final Results]")
print(
    f"Deep Autoencoder Reconstruction Loss (MAE): {autoencoder.evaluate(X, X, verbose=0):.4f}"
)
print(f"User-level Recall@10: {recall_at_10:.2%}")
print(f"Matched {hits} users out of {len(unique_anchors)} total anchors in edgelist.")

Epoch 1/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.2819 - val_loss: 0.2422
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.1678 - val_loss: 0.1151
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1082 - val_loss: 0.0913
Epoch 4/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0967 - val_loss: 0.0905
Epoch 5/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0885 - val_loss: 0.0837
Epoch 6/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0884 - val_loss: 0.0730
Epoch 7/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0780 - val_loss: 0.0751
Epoch 8/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0755 - val_loss: 0.0700
Epoch 9/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0713 - val_loss: 0.0634
Epoch 10/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0707 - val_loss: 0.0606
Epoch 11/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0656 - val_loss: 0.0574
Epoch 12/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss:

In [11]:
# Pair level recall：
user_embeddings = encoder_model.predict(X)
sim_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

print(f"Starting Pair-level Evaluation on {len(valid_edgelist)} pairs...")

hits = 0
total_pairs = len(valid_edgelist)

for idx, row in valid_edgelist.iterrows():
    anchor = row["user_id_anchor"]
    positive = row["user_id_positive"]

    top_10_similar = similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()

    if positive in top_10_similar:
        hits += 1

pair_level_recall = hits / total_pairs

print("\n[Final Results - Pair-level]")
print(
    f"Deep Autoencoder Reconstruction Loss (MAE): {autoencoder.evaluate(X, X, verbose=0):.4f}"
)
print(f"Pair-level Recall@10: {pair_level_recall:.2%}")
print(f"Successfully found {hits} out of {total_pairs} preset pairs.")

61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Starting Pair-level Evaluation on 13335 pairs...

[Final Results - Pair-level]
Deep Autoencoder Reconstruction Loss (MAE): 0.0265
Pair-level Recall@10: 1.96%
Successfully found 262 out of 13335 preset pairs.
